# Werkcollege-opdrachten Week 1.3

## Dependencies importeren

Kopieer in het codeblok hieronder van het vorige practicum de import-code voor de dependencies die het vaakst worden gebruikt om data in te lezen. Geef er ook de gebruikelijke aliassen aan.<br>
Zet eventuele warnings uit.

In [76]:
import warnings
warnings.simplefilter('ignore')
import pandas
import sqlite3

Zet het bestand go_sales_train.sqlite in een makkelijk te vinden map

## Databasetabellen inlezen

Kopieer in het codeblok hieronder van het vorige practicum de code om een connectie met het bestand go_sales_train.sqlite te maken.

In [77]:
conn = sqlite3.connect("..\Data\Week1\go_sales_train.sqlite")
print(conn)

Lees van de ingelezen go_sales_train-database te volgende tabellen in met behulp van "SELECT * FROM *tabel*".
- product
- product_type
- product_line
- sales_staff
- sales_branch
- retailer_site
- country
- order_header
- order_details
- returned_item
- return_reason

In [78]:
tables_query = "SELECT name FROM sqlite_master WHERE type= 'table';"
tables = pandas.read_sql(tables_query, conn)

df = {}

for table_name in tables['name']:
    query = f"SELECT * FROM {table_name}"
    df[table_name] = pandas.read_sql(query, conn)

Krijg je een "no such table" error? Dan heb je misschien met .connect() per ongeluk een leeg  databasebestand (.sqlite) aangemaakt. <u>Let op:</u> lees eventueel de informatie uit het Notebook van werkcollege 1.1b nog eens goed door.

Als je tijdens onderstaande opdrachten uit het oog verliest welke tabellen er allemaal zijn, kan je deze Pythoncode uitvoeren:

In [79]:
sql_query = "SELECT name FROM sqlite_master WHERE type='table';"
#Vul dit codeblok verder in
pandas.read_sql(sql_query, conn)
#Op de puntjes hoort de connectie naar go_sales_train óf go_staff_train óf go_crm_train te staan.

,name
0,country
1,order_details
2,order_header
3,order_method
4,product
5,product_line
6,product_type
7,retailer_site
8,return_reason
9,returned_item


erachter 

Let op! Voor alle onderstaande opdrachten mag je <u>alleen Python</u> gebruiken, <u>geen SQL!</u>

## Selecties op één tabel zonder functies

Geef een overzicht met daarin de producten en hun productiekosten waarvan de productiekosten lager dan 100 dollar en hoger dan 50 dollar ligt. (2 kolommen, 23 rijen)

In [80]:
table_name = tables['name'][4]
table_data = df[table_name]

production_costs = (table_data['PRODUCTION_COST'] > 50) & (table_data['PRODUCTION_COST'] < 100)
table_data.loc[production_costs, ['PRODUCT_NAME', 'PRODUCTION_COST']]

,PRODUCT_NAME,PRODUCTION_COST
5,TrailChef Deluxe Cook Set,85.11
7,TrailChef Double Flame,75.00
16,Hibernator Lite,60.00
17,Hibernator,86.00
19,Hibernator Self - Inflating Mat,78.55
22,Hibernator Camp Cot,65.33
23,Canyon Mule Climber Backpack,62.50
45,Granite Climbing Helmet,52.86
47,Husky Harness Extreme,55.00
57,Granite Hammer,57.13


Geef een overzicht met daarin de producten en hun marge waarvan de marge lager dan 20 % of hoger dan 60 % ligt. (2 kolommen, 7 rijen) 

In [81]:
margin = (table_data['MARGIN'] < 0.20) | (table_data['MARGIN'] > 0.60)
table_data.loc[margin, ['PRODUCT_NAME', 'MARGIN']]

,PRODUCT_NAME,MARGIN
20,Hibernator Pad,0.17
23,Canyon Mule Climber Backpack,0.17
85,BugShield Natural,0.67
86,BugShield Spray,0.67
87,BugShield Lotion Lite,0.70
88,BugShield Lotion,0.63
89,BugShield Extreme,0.63


Geef een overzicht met daarin de landen waar met francs wordt betaald. Sorteer de uitkomst op land.  (1 kolom, 3 rijen)

In [82]:
table_name = tables['name'][0]
table_data = df[table_name]

currency = table_data['CURRENCY_NAME'] == 'francs'
table_data.loc[currency, ['COUNTRY']]

,COUNTRY
0,France
7,Switzerland
15,Belgium


Geef een overzicht met daarin de verschillende introductiedatums waarop producten met meer dan 50% marge worden geïntroduceerd (1 kolom, 7 rijen) 

In [83]:
table_name = tables['name'][4]
table_data = df[table_name]

introduction_dates = table_data['MARGIN'] > 0.50
table_data.loc[introduction_dates, ['INTRODUCTION_DATE']].drop_duplicates('INTRODUCTION_DATE')

,INTRODUCTION_DATE
66,2000-10-26
85,1995-02-15
101,2003-12-10
103,2003-12-18
104,2003-12-27
106,2004-01-13
110,2003-12-15


Geef een overzicht met daarin het eerste adres en de stad van verkoopafdelingen waarvan zowel het tweede adres als de regio bekend is (2 kolommen, 7 rijen)

In [84]:
table_name = tables['name'][10]
table_data = df[table_name]

chujio = table_data['REGION'].notna() & table_data['ADDRESS2'].notna()
table_data.loc[chujio, ['ADDRESS1', 'CITY']]

,ADDRESS1,CITY
2,Singelgravenplein 4,Amsterdam
13,Prol. Paseo de la Reforma No. 51,Distrito Federal
14,202-2-3 Hyakunincho,Tokyo
15,543-225 Asahi,Osaka City
16,2315 Queen's Ave,Melbourne
18,"Avenida Paulista, 333",São Paulo
24,3 Albany Court,Birmingham


Geef een overzicht met daarin de landen waar met dollars (dollars of new dollar) wordt betaald. Sorteer de uitkomst op land. (1 kolom, 4 rijen) 

In [87]:
table_name = tables['name'][0]
table_data = df[table_name]

chujio = table_data['CURRENCY_NAME'].str.contains('dollar')
table_data.loc[chujio, ['COUNTRY']].sort_values(by='COUNTRY', ascending=True)

,COUNTRY
14,Australia
3,Canada
11,Taiwan
2,United States


Geef een overzicht met daarin beide adressen en de stad van vestigingen van klanten waarvan de postcode begint met een ‘D’ (van duitsland). Filter op vestigingen die een tweede adres hebben. (3 kolommen, 2 rijen) 

In [91]:
table_name = tables['name'][7]
table_data = df[table_name]

chujio = (table_data['POSTAL_ZONE'].str[0] == 'D') & table_data['ADDRESS2'].notna()
table_data.loc[chujio, ['ADDRESS1', 'ADDRESS2', 'CITY']]

,ADDRESS1,ADDRESS2,CITY
31,Röntgenstraße 90,3. Tür rechts,Frankfurt
59,Grubesallee 141,4. Stock,Hamburg


## Selecties op één tabel met functies

Geef het totaal aantal producten dat is teruggebracht (1 waarde) 

In [99]:
table_name = tables['name'][9]
table_data = df[table_name]

table_data['RETURN_QUANTITY']['Hoeveelheid'].sum()

KeyError: 'Hoeveelheid'

Geef het aantal regio’s waarin verkoopafdelingen gevestigd zijn. (1 waarde)

Maak 3 variabelen:
- Een met de laagste
- Een met de hoogste
- Een met de gemiddelde (afgerond op 2 decimalen)

marge van producten (3 kolommen, 1 rij) 

Geef het aantal vestigingen van klanten waarvan het 2e adres niet bekend is (1 waarde)

Geef de gemiddelde kostprijs van de verkochte producten waarop korting (unit_sale_price < unit_price) is verleend (1 waarde) 

Geef een overzicht met daarin het aantal medewerkers per medewerkersfunctie (2 kolommen, 7 rijen) 

Geef een overzicht met daarin per telefoonnummer het aantal medewerkers dat op dat telefoonnummer bereikbaar is. Toon alleen de telefoonnummer waarop meer dan 4 medewerkers bereikbaar zijn. (2 kolommen, 10 rijen) 

## Selecties op meerdere tabellen zonder functies

Geef een overzicht met daarin het eerste adres en de stad van vestigingen van klanten uit ‘Netherlands’ (2 kolommen, 20 rijen) 

Geef een overzicht met daarin de productnamen die tot het producttype ‘Eyewear’ behoren. (1 kolom, 5 rijen) 

Geef een overzicht met daarin alle unieke eerste adressen van klantvestigingen en de voornaam en achternaam van de verkopers die ‘Branch Manager’ zijn en aan deze vestigingen hebben verkocht (3 kolommen, 1 rij) 

Geef een overzicht met daarin van de verkopers hun functie en indien zij iets hebben verkocht de datum waarop de verkoop heeft plaatsgevonden. Laat alleen de verschillende namen van de posities zien van de verkopers die het woord ‘Manager’ in hun positienaam hebben staan. (2 kolommen, 7 rijen) 

Geef een overzicht met daarin de verschillende namen van producten en bijbehorende namen van producttypen van de producten waarvoor ooit meer dan 750 stuks tegelijk verkocht zijn. (2 kolommen, 9 rijen) 

Geef een overzicht met daarin de productnamen waarvan ooit meer dan 40% korting is verleend. De formule voor korting is: (unit_price - unit_sale_price) / unit_price (1 kolom, 8 rijen) 

Geef een overzicht met daarin de retourreden van producten waarvan ooit meer dan 90% van de aangeschafte hoeveelheid is teruggebracht (return_quantity/quantity). (1 kolom, 3 rijen) 

## Selecties op meerdere tabellen met functies

Geef een overzicht met daarin per producttype het aantal producten die tot dat producttype behoren. (2 kolommen, 21 rijen) 

Geef een overzicht met daarin per land het aantal vestigingen van klanten die zich in dat land bevinden. (2 kolommen, 21 rijen) 

Geef een overzicht met daarin van de producten behorend tot het producttype ‘Cooking Gear’ per productnaam de totaal verkochte hoeveelheid en de gemiddelde verkoopprijs. Sorteer de uitkomst op totaal verkochte hoeveelheid. (4 kolommen, 10 rijen) 

Geef een overzicht met daarin per land de naam van het land, de naam van de stad waar de verkoopafdeling is gevestigd (noem de kolomnaam in het overzicht ‘verkoper’) en het aantal steden waar zich klanten bevinden in dat land (noem de kolomnaam in het overzicht ‘klanten’) (3 kolommen, 29 rijen) 

## Pythonvertalingen van SUBSELECT en UNION met o.a. for-loops

Geef een overzicht met daarin de voornaam en de achternaam van de medewerkers die nog nooit wat hebben verkocht (2 kolommen, 25 rijen) 

Geef een overzicht met daarin het aantal producten waarvan de marge lager is dan de gemiddelde marge van alle producten samen. Geef in het overzicht tevens aan wat de gemiddelde marge is van dit aantal producten waarvan de marge lager dan de gemiddelde marge van alle producten samen is. (1 kolom, 2 rijen) 

Geef een overzicht met daarin de namen van de producten die voor meer dan 500 (verkoopprijs) zijn verkocht maar nooit zijn teruggebracht. (1 kolom, 13 rijen) 

Geef een overzicht met daarin per (achternaam van) medewerker of hij/zij manager is of niet, door deze informatie toe te voegen als extra 'Ja/Nee'-kolom.<br>
Hint: gebruik een for-loop waarin je o.a. bepaalt of het woord 'Manager' in de functie (position_en) staat. (2 kolommen, 102 rijen).

Met de onderstaande code laat je Python het huidige jaar uitrekenen.

In [ ]:
from datetime import date
date.today().year

2025

Met de onderstaande code selecteer je op een bepaald jaartal uit een datum.

In [ ]:
from datetime import datetime

date_str = '16-8-2013'
date_format = '%d-%m-%Y'
date_obj = datetime.strptime(date_str, date_format)

date_obj.year

2013

Geef met behulp van bovenstaande hulpcode een overzicht met daarin op basis van het aantal jaar dat iemand in dienst is of een medewerker ‘kort in dienst’ (minder dan 25 jaar in dienst) of een ‘lang in dienst’ (groter gelijk dan 12 jaar in dienst) is. Geef daarbij per medewerker in een aparte kolom zowel ‘kort in dienst’ als ‘lang in dienst’ aan. Gebruik (wederom) een for-loop.<br>
(2 kolommen, 102 rijen) 

## Van Jupyter Notebook naar Pythonproject

1. Richt de map waarin jullie tot nu toe hebben gewerkt in volgens de mappenstructuur uit de slides.
2. Maak van de ontstane mappenstructuur een Pythonproject dat uitvoerbaar is vanuit de terminal. Maak daarin een .py-bestand dat minstens 5 antwoorden uit dit notebook (in de vorm van een DataFrame) exporteert naar Excelbestanden. Alle notebooks mogen als notebook blijven bestaan.
3. Zorg ervoor dat dit Pythonproject zijn eigen repo heeft op Github. Let op: je virtual environment moet <b><u>niet</u></b> meegaan naar Github.

Je mag tijdens dit proces je uit stap 1 ontstane mappenstructuur aanpassen, zolang je bij het beoordelingsmoment kan verantwoorden wat de motivatie hierachter is. De slides verplichten je dus nergens toe.